# Gas lasers and longitudinal modes {#sec-gas}

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippevelha/Optoelectronics-teaching/blob/main/chapter4/notebooks/gas-lasers.ipynb){.colab-link} &nbsp;Run this chapter's code in Google Colab (no installation).

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippevelha/Optoelectronics-teaching/blob/main/chapter4/notebooks/gas-lasers.ipynb)

> Generated from the Quarto companion. The browser sliders of the website are not included; see `interactive_explorers.ipynb` for ipywidgets versions.

In [ ]:
# --- setup: works locally and in Google Colab ---------------------------
import sys
if 'google.colab' in sys.modules:
    import urllib.request
    urllib.request.urlretrieve('https://raw.githubusercontent.com/philippevelha/Optoelectronics-teaching/main/chapter4/laserlib.py', 'laserlib.py')
else:
    sys.path.insert(0, '..')                 # laserlib.py is in the parent folder
def ojs_define(**kw): pass                   # only used by the Quarto website
# ----------------------------------------------------------------------
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import laserlib as lz
lz.use_style()

*Slides 45–53 (the He–Ne laser: principle, modes, other lasing lines, efficiency example), 54–60 (Doppler broadening and the
mode-counting example) and 61–62 (the argon-ion laser).*

## Key relations

In the He–Ne laser (slides 45–47) an electrical discharge excites helium atoms into the metastable $2^1S_0$ and $2^3S_1$ states.
Their energies (20.61 and 19.82 eV) almost coincide with the Ne $3s_2$ and $2s_2$ levels, so a collision hands the energy over,
$\text{He}^*+\text{Ne}\to\text{He}+\text{Ne}^*$. The excited Ne atoms are inverted with respect to the short-lived $2p$ levels.
The $3s_2\to2p_4$ transition gives the red line at 632.8 nm; other transitions give the green, yellow, orange and infrared
lines of slide 52.

The two mirrors at distance $L$ form a Fabry–Perot resonator. Only standing waves with a whole number of half wavelengths fit
(slides 48–49):

$$
m\frac{\lambda_m}{2n}=L,\qquad \nu_m=m\frac{c}{2nL},\qquad \Delta\nu_m=\frac{c}{2nL},\qquad \Delta\lambda_m\approx\frac{\lambda^2}{2nL}.
$$

The output spectrum is the product of these **longitudinal modes** with the **optical gain curve** of the medium. In a gas the
gain curve is Doppler broadened: atoms moving toward the mirror see a blue-shifted frequency, atoms moving away a red-shifted one.
The Maxwell velocity distribution makes the curve Gaussian with FWHM (slide 49)

$$
\Delta\nu_{1/2}=2\nu_0\sqrt{\frac{2k_BT\ln2}{Mc^2}},\qquad \Delta\lambda_{1/2}\approx\Delta\nu_{1/2}\frac{\lambda}{\nu}.
$$

## Gain curve and cavity modes (slides 48–50)

The panel draws the Doppler gain curve and the cavity modes. Modes whose gain exceeds the loss line (threshold, @sec-threshold)
can lase.

```{ojs}
//| echo: false
gasLines = ({"He–Ne 632.8 nm (Ne, 20.18 g/mol)": [632.8, 20.18], "He–Ne 543.5 nm (green)": [543.5, 20.18],
             "He–Ne 1523 nm (IR)": [1523.1, 20.18], "Ar⁺ 488.0 nm (39.95 g/mol)": [488.0, 39.95],
             "Ar⁺ 514.5 nm": [514.5, 39.95], "CO₂ 10.6 µm (44 g/mol)": [10600, 44.0]})
viewof gl = Inputs.form({
  line: Inputs.select(Object.keys(gasLines), {value: "He–Ne 632.8 nm (Ne, 20.18 g/mol)", label: "laser line"}),
  T: Inputs.range([200, 2000], {value: 400, step: 10, label: "gas temperature T (K)"}),
  L: Inputs.range([5, 200], {value: 40, step: 1, label: "cavity length L (cm)"}),
  thr: Inputs.range([0, 0.99], {value: 0.5, step: 0.01, label: "threshold / peak gain"}),
  shift: Inputs.range([0, 1], {value: 0, step: 0.01, label: "mode position (fraction of Δν_m)"})
})
```

```{ojs}
//| echo: false
glOut = {
  const [lam0, Mg] = gasLines[gl.line], c = 299792458, k = 1.380649e-23, M = Mg*1e-3/6.02214076e23;
  const nu0 = c/(lam0*1e-9), fw = 2*nu0*Math.sqrt(2*k*gl.T*Math.log(2)/(M*c*c)), dnm = c/(2*gl.L*1e-2);
  const G = f => Math.exp(-4*Math.log(2)*(f/fw)**2), span = 1.6*fw, curve = [], modes = [];
  for (let i = 0; i <= 400; i++) { const f = -span + 2*span*i/400; curve.push({f: f/1e9, g: G(f)}); }
  const m0 = Math.round(-span/dnm) - 1, m1 = Math.round(span/dnm) + 1;
  if (m1 - m0 < 4000) for (let m = m0; m <= m1; m++) { const f = (m + gl.shift)*dnm; if (Math.abs(f) <= span) modes.push({f: f/1e9, g: G(f), lase: G(f) > gl.thr}); }
  const nLase = modes.filter(d => d.lase).length;
  return {fw, dnm, curve, modes, nLase, lam0, nu0, dlam: fw*lam0/nu0, mnum: 2*gl.L*1e-2/(lam0*1e-9)};
}
Plot.plot({
  width: Math.min(width, 1050), height: 300, x: {label: "frequency offset ν − ν₀ (GHz)"}, y: {label: "gain / peak gain", domain: [0, 1.05], grid: true},
  marks: [Plot.areaY(glOut.curve, {x: "f", y: "g", fill: "#eb6834", fillOpacity: 0.15}), Plot.line(glOut.curve, {x: "f", y: "g", stroke: "#eb6834", strokeWidth: 2}),
          Plot.ruleX(glOut.modes, {x: "f", y1: 0, y2: "g", stroke: d => d.lase ? "#2a78d6" : "#c3c2b7", strokeWidth: glOut.modes.length > 200 ? 1 : 2.5}),
          Plot.ruleY([gl.thr], {stroke: "#e34948", strokeDasharray: "5,3"})]
})
```

```{ojs}
//| echo: false
html`<div class="result-box">Doppler FWHM Δν<sub>1/2</sub> = <b>${(glOut.fw/1e9).toFixed(3)} GHz</b> (Δλ<sub>1/2</sub> = ${(glOut.dlam*1e3).toPrecision(3)} pm) ·
mode spacing c/2L = <b>${(glOut.dnm/1e6).toFixed(1)} MHz</b> · mode number m ≈ ${glOut.mnum.toExponential(4)} ·
modes above threshold: <b>${glOut.nLase}</b> (estimate Δν<sub>1/2</sub>/Δν<sub>m</sub> = ${(glOut.fw/glOut.dnm).toFixed(1)})</div>`
```

Two consequences show up immediately:

* The number of lasing modes depends on where the comb sits under the gain curve ("mode position"). Moving the comb by half a
  spacing changes the count by one. In a real laser the cavity length drifts thermally by many wavelengths, so the modes slide
  under the curve and the output power fluctuates ("mode sweeping").
* A cavity shorter than about $c/(2\Delta\nu_{1/2})\approx10$ cm keeps a single He–Ne mode above threshold. Single-frequency
  He–Ne lasers are short, and their output is low.

## Worked example: Doppler-broadened linewidth (slides 55–60), recomputed

> He–Ne laser at 632.8 nm, discharge temperature 127 °C, Ne atomic mass 20.2 g/mol, tube length 40 cm. Find the Doppler
> linewidths, the mode number, the mode separation and the number of modes within the linewidth.

In [ ]:
#| label: tbl-doppler
#| tbl-cap: "Slides 55–60 recomputed. The rms estimate uses Δν_rms ≈ 2ν₀v_x/c with ½Mv_x² = ½k_BT."
lam0, T, L = 632.8e-9, 400.0, 0.40
nu0 = lz.c0/lam0
M = 20.2e-3/lz.NA
vx = np.sqrt(lz.kB*T/M)
d_rms = 2*nu0*vx/lz.c0
d_fwhm = lz.doppler_fwhm(nu0, T, 20.2)
dnu_m, dlam_m = lz.mode_spacing(L, lam0)
rows = [("rms velocity v_x", f"{vx:.1f} m/s", "405.9 m/s"),
        ("Δν_rms ≈ 2ν₀v_x/c", f"{d_rms/1e9:.3f} GHz", "1.28 GHz"),
        ("Doppler FWHM Δν_1/2", f"{d_fwhm/1e9:.3f} GHz ({d_fwhm/d_rms - 1:.0%} wider)", "1.51 GHz (18 % wider)"),
        ("Δλ_1/2", f"{d_fwhm*lam0/nu0*1e12:.2f} pm", "2.02 pm"),
        ("mode number m₀ = 2L/λ₀", f"{2*L/lam0:,.1f}", "1,264,222.5"),
        ("mode spacing Δλ_m, Δν_m", f"{dlam_m*1e12:.3f} pm, {dnu_m/1e6:.0f} MHz", "0.501 pm, 375 MHz"),
        ("modes within Δν_1/2", f"{d_fwhm/dnu_m:.2f}", "≈ 4, at most 4 to 5")]
pd.DataFrame(rows, columns=["quantity", "computed", "slide"])

All the values match the slides. The FWHM exceeds the rms estimate because the FWHM of a Gaussian is $2\sqrt{2\ln2}=2.35$
standard deviations, while $2v_x$ corresponds to two standard deviations.

In [ ]:
#| label: fig-modes
#| fig-cap: "The He–Ne example: Doppler gain curve (1.51 GHz FWHM) and the cavity modes spaced by 375 MHz (L = 40 cm). With the threshold at half the peak gain, four modes lase (blue), as in slide 60."
f = np.linspace(-2.5e9, 2.5e9, 1000)
g = np.exp(-4*np.log(2)*(f/d_fwhm)**2)
fm = (np.arange(-8, 9) + 0.3)*dnu_m
gm = np.exp(-4*np.log(2)*(fm/d_fwhm)**2)
fig, ax = plt.subplots(figsize=(7.5, 3.4))
ax.fill_between(f/1e9, g, color=lz.PALETTE[1], alpha=0.15); ax.plot(f/1e9, g, color=lz.PALETTE[1])
for x, y in zip(fm, gm):
    ax.vlines(x/1e9, 0, y, color=lz.PALETTE[0] if y > 0.5 else lz.GRID, lw=2.5)
ax.axhline(0.5, color=lz.PALETTE[7], ls="--", lw=1.2)
ax.set(xlabel="ν − ν₀ (GHz)", ylabel="gain / peak", ylim=(0, 1.05)); plt.show()

## Worked example: efficiency of the He–Ne laser (slide 53)

In [ ]:
#| label: tbl-hene-eff
#| tbl-cap: "Slide 53: a 5 mW He–Ne tube at 2000 V and 7 mA, and the Ar⁺ laser for comparison (typical values)."
pd.DataFrame([
    {"laser": "He–Ne, 632.8 nm (slide 53)", "P_out": "5 mW", "electrical input": "2000 V × 7 mA = 14 W",
     "efficiency": f"{5e-3/14:.3%}", "intensity in a 1 mm beam": f"{5e-3/(np.pi*0.5e-3**2)/1e3:.1f} kW/m²"},
    {"laser": "Ar⁺, 488 + 514.5 nm (typical)", "P_out": "5 W", "electrical input": "≈ 200 V × 40 A = 8 kW",
     "efficiency": f"{5/8e3:.3%}", "intensity in a 1 mm beam": f"{5/(np.pi*0.5e-3**2)/1e6:.1f} MW/m²"}])

The He–Ne laser converts less than 0.1 % of its electrical power into light. The Ar⁺ laser is worse: the laser levels sit 35 eV
above the neutral ground state (slide 62), while each photon carries only 2.5 eV, and kilowatts of heat must be carried away by
water cooling. Both lasers have been replaced by laser diodes and diode-pumped solid-state lasers in most applications. The
He–Ne survives where its spectral purity and wavelength accuracy matter (interferometry, metrology).

::: {.callout-tip collapse="true"}
## Going further
- **Mode competition in an inhomogeneous medium.** Each longitudinal mode interacts with its own velocity class of atoms, at
  $\pm v=c(\nu_m-\nu_0)/\nu_0$. Modes do not steal each other's atoms, so several modes lase together. Compare with the
  homogeneously broadened Nd:YAG laser in @sec-broadening.
- **Lamb dip.** A mode tuned exactly to $\nu_0$ interacts with the $v=0$ atoms in both directions of travel, so it has *fewer*
  atoms available than a slightly detuned mode. The output power versus cavity tuning shows a dip at line centre, used to
  stabilise He–Ne lasers to about $10^{-9}$.
- **Brewster windows.** The tube windows are tilted at Brewster's angle (Ch. 1): the p polarisation passes without loss, the s
  polarisation loses about 15 % per pass. Compute the round-trip loss of each polarisation with the Fresnel equations and show why
  the output is linearly polarised.
- **Isotope shift.** ²⁰Ne and ²²Ne lines differ by about 900 MHz at 632.8 nm. Commercial tubes use a mixture to broaden the gain
  curve. Model the gain as the sum of two Gaussians.
:::

::: {.callout-note collapse="true"}
## Complements
- A. Javan, W. R. Bennett and D. R. Herriott, "Population inversion and continuous optical maser oscillation in a gas discharge
  containing a He–Ne mixture", *Phys. Rev. Lett.* 6, 106 (1961).
- Kasap, sec. 4.5; Siegman, *Lasers*, ch. 10 (Doppler broadening) and 31 (Lamb dip).
:::

## Try it yourself

1. Compute the Doppler FWHM of the CO₂ laser line at 10.6 µm (400 K). How many modes does a 1 m cavity support?
2. Why is the Doppler width of the Ar⁺ lines (5–6 GHz) much larger than that of the He–Ne laser, although Ar is twice as heavy?
   (Hint: the discharge temperature.)
3. A He–Ne laser has $L=20$ cm. What is the largest and smallest number of modes above the half-maximum threshold?
4. Show that $\Delta\lambda_m\approx\lambda^2/2nL$ follows from $m\lambda_m/2n=L$, and estimate the error of the approximation for
   the slide example.